# Question 1: Campaign Spending Trends
**Ohio 2024 U.S. Senate race: Sherrod Brown (Democrat) vs Bernie Moreno (Republican). FEC data, 2023-24 cycle.**

**Question:** How does campaign spending change over the 2023-24 cycle, and how do the two campaigns differ?

**What this notebook answers:** How big Brown's campaign spending advantage was, when it opened up, and how both campaigns spent over time. Only the campaigns' own spending is used here. Outside groups are covered in Question 3.

## Setup and loading the data

In [1]:
# Imports and output folder
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT = "../outputs/question_1/"
os.makedirs(OUT, exist_ok=True)
pd.options.display.float_format = "{:,.2f}".format

In [2]:
# Load the cleaned spending file and keep only the rows used in the analysis
spend = pd.read_csv("../data/processed/campaign_spending.csv", parse_dates=["date"])
committees = pd.read_csv("../data/processed/dim_committees.csv")
print(len(spend), "rows loaded;", len(committees), "committees in dim_committees.csv")

df = spend[spend["include"] == True].copy()
print(len(df), "rows have include == True")

df["who"] = df["candidate"].map({"Sherrod Brown": "Brown", "Bernie Moreno": "Moreno"})

10718 rows loaded; 9 committees in dim_committees.csv
5912 rows have include == True


In [3]:
# Group the 16 categories into three spending types
VOTER = ["TV, radio & other media", "Digital advertising", "Direct mail & printing",
         "Texting & phone", "Canvassing & field"]
df["spending_type"] = "Staff, overhead and other"
df.loc[df["category"].isin(VOTER), "spending_type"] = "Voter communication"
df.loc[df["category"] == "Fundraising costs", "spending_type"] = "Fundraising costs"
df["spending_type"].value_counts()

spending_type
Staff, overhead and other    4434
Voter communication          1149
Fundraising costs             329
Name: count, dtype: int64

In [4]:
# The five time periods and the key dates
phases = pd.DataFrame({
    "phase": ["2023",
              "Jan 1 - Mar 19, 2024 (before the primary)",
              "Mar 20 - Aug 31, 2024",
              "Sep 1 - Nov 5, 2024 (final stretch)",
              "After Election Day"],
    "start": pd.to_datetime(["2023-01-01", "2024-01-01", "2024-03-20", "2024-09-01", "2024-11-06"]),
    "end":   pd.to_datetime(["2023-12-31", "2024-03-19", "2024-08-31", "2024-11-05", "2024-12-31"]),
})
phases["days"] = (phases["end"] - phases["start"]).dt.days + 1

edges = pd.to_datetime(["2022-12-31", "2023-12-31", "2024-03-19", "2024-08-31", "2024-11-05", "2024-12-31"])
df["phase"] = pd.cut(df["date"], bins=edges, labels=phases["phase"]).astype(str)
assert df["phase"].isin(phases["phase"]).all(), "some payments fall outside the five phases"

key_dates = pd.DataFrame({
    "event": ["Ohio primary", "Republican convention", "Democratic convention", "Labor Day", "Election Day"],
    "date": pd.to_datetime(["2024-03-19", "2024-07-15", "2024-08-19", "2024-09-02", "2024-11-05"]),
    "description": ["Ohio primary election, 19 Mar 2024",
                    "Republican National Convention, 15-18 Jul 2024 (starts on day 1)",
                    "Democratic National Convention, 19-22 Aug 2024 (starts on day 1)",
                    "Labor Day, 2 Sep 2024 (start of the fall campaign)",
                    "Election Day, 5 Nov 2024"]})
phases

,phase,start,end,days
0,2023,2023-01-01,2023-12-31,365
1,"Jan 1 - Mar 19, 2024 (before the primary)",2024-01-01,2024-03-19,79
2,"Mar 20 - Aug 31, 2024",2024-03-20,2024-08-31,165
3,"Sep 1 - Nov 5, 2024 (final stretch)",2024-09-01,2024-11-05,66
4,After Election Day,2024-11-06,2024-12-31,56


In [5]:
# Colours and the campaign-only view (the candidates' own committees)
BROWN, MORENO = "#1f5fa8", "#c0392b"
COLORS = {"Brown": BROWN, "Moreno": MORENO}
PHASE_COLORS = ["#999999", "#E69F00", "#56B4E9", "#009E73", "#CC79A7"]
PHASE_WRAP = {p: p.replace(" (", "<br>(") for p in phases["phase"]}
PHASE_SHORT = dict(zip(phases["phase"], ["2023", "Jan 1 -<br>Mar 19", "Mar 20 -<br>Aug 31", "Sep 1 -<br>Nov 5", "After<br>Nov 5"]))

campaign = df[df["group"] == "campaign"].copy()
print(len(campaign), "campaign-group payments;", len(df) - len(campaign), "affiliated-group payments")

4700 campaign-group payments; 1212 affiliated-group payments


In [6]:
# Shared chart style and a helper to save tables
def style_chart(fig, title, subtitle, xtitle=None, ytitle=None, money=False,
                height=520, bottom=90, right=120):
    fig.update_layout(
        template="plotly_white", height=height, font={"size": 12},
        title={"text": title + "<br><sup>" + subtitle + "</sup>", "font": {"size": 18},
               "xref": "paper", "x": 0, "xanchor": "left", "y": 0.95, "yanchor": "top"},
        margin={"l": 70, "r": right, "t": 120, "b": bottom},
        legend={"orientation": "v", "x": 1.02, "xanchor": "left", "y": 1, "yanchor": "top"})
    fig.update_xaxes(title_text=xtitle, title_font={"size": 12}, tickfont={"size": 12}, showgrid=False)
    fig.update_yaxes(title_text=ytitle, title_font={"size": 12}, tickfont={"size": 12})
    if money:
        fig.update_yaxes(tickprefix="$", ticksuffix=" M", tickformat=".1f")
    return fig

def add_key_dates(fig):
    for date, label, anchor in [("2024-03-19", "Primary 19 Mar", "center"), ("2024-11-05", "Election Day 5 Nov", "right")]:
        fig.add_shape(type="line", xref="x", yref="paper", x0=date, x1=date, y0=0, y1=1,
                      line={"color": "#555555", "width": 1, "dash": "dot"})
        fig.add_annotation(x=date, y=1, xref="x", yref="paper", text=label, showarrow=False,
                           xanchor=anchor, yanchor="bottom", font={"size": 11, "color": "#555555"})
    return fig

def save_table(table, filename):
    table = table.copy()
    for col in table.columns:
        if str(table[col].dtype).startswith("datetime"):
            table[col] = table[col].dt.strftime("%Y-%m-%d")
    table.round(4).to_csv(OUT + filename, index=False)
    print("saved", filename, table.shape)

In [7]:
# Save the period and key-date definitions
save_table(phases, "phase_definitions.csv")
save_table(key_dates, "key_dates.csv")

saved phase_definitions.csv (5, 4)
saved key_dates.csv (5, 3)


## Metric 1: Total spent
**Question:** How much did each campaign spend in total, and how do they compare?
**Definition:** Total spent = sum of `amount`, `include == True`, campaign committees (2023-01-01 to 2024-12-31). The table also shows the affiliated committees and "all committees" for reference.

In [8]:
# Total spent by candidate: campaign, affiliated and all committees
by_group = df.groupby(["who", "group"]).agg(total_spent=("amount", "sum"), payment_count=("amount", "size")).reset_index()
by_group = by_group.rename(columns={"group": "scope"})
all_committees = df.groupby("who").agg(total_spent=("amount", "sum"), payment_count=("amount", "size")).reset_index()
all_committees["scope"] = "all committees"

total_table = pd.concat([by_group, all_committees]).rename(columns={"who": "candidate"})
moreno_total = total_table[total_table["candidate"] == "Moreno"].set_index("scope")["total_spent"]
total_table["ratio_to_moreno"] = total_table["total_spent"] / total_table["scope"].map(moreno_total)
total_table = total_table[["candidate", "scope", "total_spent", "payment_count", "ratio_to_moreno"]]
save_table(total_table, "total_spent.csv")
total_table

saved total_spent.csv (6, 5)


,candidate,scope,total_spent,payment_count,ratio_to_moreno
0,Brown,affiliated,"3,006,670.58",744,0.70
1,Brown,campaign,"97,460,584.55",2517,3.77
2,Moreno,affiliated,"4,302,910.40",468,1.00
3,Moreno,campaign,"25,820,920.56",2183,1.00
0,Brown,all committees,"100,467,255.13",3261,3.34
1,Moreno,all committees,"30,123,830.96",2651,1.00


In [9]:
# Campaign totals and the Brown to Moreno ratio
camp = total_table[total_table["scope"] == "campaign"].set_index("candidate")
ratio_total = camp.loc["Brown", "total_spent"] / camp.loc["Moreno", "total_spent"]

In [10]:
# Chart 1: total spent
fig = go.Figure()
for who, left, right in [("Brown", 0.0, 0.46), ("Moreno", 0.54, 1.0)]:
    fig.add_shape(type="rect", xref="paper", yref="paper", x0=left, x1=right, y0=0.32, y1=1.0,
                  fillcolor="#f7f9fc", line={"color": "#e3e8ef", "width": 1}, layer="below")
    fig.add_trace(go.Indicator(
        mode="number", value=camp.loc[who, "total_spent"] / 1e6,
        number={"prefix": "$", "suffix": " M", "valueformat": ",.1f", "font": {"color": COLORS[who], "size": 70}},
        title={"text": "<b>" + who + "</b>: total spent", "font": {"size": 18, "color": COLORS[who]}},
        domain={"x": [left, right], "y": [0.46, 0.96]}))
    fig.add_annotation(x=(left + right) / 2, y=0.43, xref="paper", yref="paper", showarrow=False, xanchor="center", yanchor="top",
                       text="exactly $" + format(camp.loc[who, "total_spent"], ",.2f"), font={"size": 13, "color": "#555555"})
fig.add_annotation(x=0.5, y=0.1, xref="paper", yref="paper", showarrow=False, xanchor="center", yanchor="middle",
                   text="<b>Brown spent " + format(ratio_total, ".1f") + " times as much as Moreno</b>",
                   font={"size": 20, "color": "#333333"}, bgcolor="#f2f2f2", borderpad=10)
style_chart(fig, "Brown's campaign spent " + format(ratio_total, ".1f") + " times as much as Moreno's",
            "Total spent, campaign committees only, $ millions, 2023-2024", height=460, bottom=70)
fig.write_html(OUT + "01_total_spent.html", include_plotlyjs="cdn")
fig.show()

## Metric 2: Spending by month (switch to quarter)
**Question:** How did each campaign's spending change over time?
**Definition:** Total spent (campaign committees) added up by payment month, or by calendar quarter. Months or quarters with no payments count as $0.

In [11]:
# Spending by month and by quarter (empty months count as 0)
all_months = pd.date_range("2023-01-01", "2024-12-01", freq="MS").strftime("%Y-%m")
all_quarters = [str(year) + "Q" + str(q) for year in (2023, 2024) for q in (1, 2, 3, 4)]

monthly = campaign.pivot_table(index="month", columns="who", values="amount", aggfunc="sum").reindex(all_months).fillna(0)
quarterly = campaign.pivot_table(index="quarter", columns="who", values="amount", aggfunc="sum").reindex(all_quarters).fillna(0)

month_to_quarter = df.drop_duplicates("month").set_index("month")["quarter"]
monthly_long = monthly.rename_axis("month").reset_index().melt(id_vars="month", var_name="candidate", value_name="total_spent")
monthly_long["quarter"] = monthly_long["month"].map(month_to_quarter)
quarterly_long = quarterly.rename_axis("quarter").reset_index().melt(id_vars="quarter", var_name="candidate", value_name="total_spent")
save_table(monthly_long[["month", "quarter", "candidate", "total_spent"]], "monthly_spend.csv")
save_table(quarterly_long, "quarterly_spend.csv")
(quarterly / 1e6).round(1).T

saved monthly_spend.csv (48, 4)
saved quarterly_spend.csv (16, 3)


quarter,2023Q1,2023Q2,2023Q3,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4
who,,,,,,,,
Brown,1.20,2.10,3.20,3.10,10.60,17.80,36.30,23.20
Moreno,0.00,0.70,0.70,3.80,3.90,1.70,7.70,7.30


In [12]:
# Chart 2: spending by month and quarter
month_dates = pd.to_datetime(monthly.index + "-01")
quarter_dates = pd.to_datetime([q[:4] + "-" + str((int(q[-1]) - 1) * 3 + 1).zfill(2) + "-01" for q in quarterly.index])
month_labels = list(month_dates.strftime("%b %Y"))
quarter_labels = [q[:4] + " Q" + q[-1] for q in quarterly.index]

fig = go.Figure()
for table, dates, labels, unit in [(monthly, month_dates, month_labels, "month"), (quarterly, quarter_dates, quarter_labels, "quarter")]:
    for who in ["Brown", "Moreno"]:
        fig.add_trace(go.Scatter(
            x=dates, y=table[who] / 1e6, name=who, mode="lines+markers", visible=(unit == "month"),
            line={"color": COLORS[who], "width": 3}, marker={"size": 7},
            text=labels, customdata=table[who].to_numpy(),
            hovertemplate=who + " (%{text}): $%{customdata:,.0f}<extra></extra>"))

fig.update_layout(hovermode="x unified", updatemenus=[{
    "type": "buttons", "direction": "right", "x": 0, "xanchor": "left", "y": 1.02, "yanchor": "bottom",
    "buttons": [{"label": "By month", "method": "update", "args": [{"visible": [True, True, False, False]}]},
                {"label": "By quarter", "method": "update", "args": [{"visible": [False, False, True, True]}]}]}])
style_chart(fig, "Both campaigns spent the most in the second half of 2024, peaking in October",
            "Total spent by payment date, campaign committees only, $ millions (quarter points sit on the first day of each quarter)",
            xtitle="Payment date", ytitle="Total spent ($ millions)", money=True)
fig.update_xaxes(range=["2022-12-01", "2025-01-15"])
add_key_dates(fig)
fig.write_html(OUT + "02_spending_over_time.html", include_plotlyjs="cdn")
fig.show()

## Metric 3: How far ahead Brown was, over time
**Question:** How big was Brown's lead in total spending on each day, and when did it open up?
**Definition:** *Lead* = Brown's cumulative total spent minus Moreno's cumulative total spent (campaign committees), counted from 1 Jan 2023 up to and including each day. *Ratio* = Brown's cumulative total divided by Moreno's.

In [13]:
# Daily spending, running total and Brown's lead
daily = campaign.pivot_table(index="date", columns="who", values="amount", aggfunc="sum")
daily = daily.reindex(pd.date_range("2023-01-01", "2024-12-31")).fillna(0)
cumulative = daily.cumsum()

lead = pd.DataFrame({"date": daily.index,
                     "brown_daily": daily["Brown"].to_numpy(), "moreno_daily": daily["Moreno"].to_numpy(),
                     "brown_cumulative": cumulative["Brown"].to_numpy(), "moreno_cumulative": cumulative["Moreno"].to_numpy()})
lead["lead"] = lead["brown_cumulative"] - lead["moreno_cumulative"]
lead["ratio"] = (lead["brown_cumulative"] / lead["moreno_cumulative"]).replace(float("inf"), float("nan"))
save_table(lead, "lead_over_time.csv")
lead.tail(3)

saved lead_over_time.csv (731, 7)


,date,brown_daily,moreno_daily,brown_cumulative,moreno_cumulative,lead,ratio
728,2024-12-29,0.00,0.00,"97,426,238.82","25,819,605.10","71,606,633.72",3.77
729,2024-12-30,"34,345.73",40.00,"97,460,584.55","25,819,645.10","71,640,939.45",3.77
730,2024-12-31,0.00,"1,275.46","97,460,584.55","25,820,920.56","71,639,663.99",3.77


In [14]:
# Lead on key days and when it first passed $25M and $50M
def first_day_over(amount):
    return lead.loc[lead["lead"] > amount, "date"].iloc[0]

milestones = pd.DataFrame({
    "milestone": ["End of 2023", "Primary (19 Mar 2024)", "Lead first passed $25M", "Lead first passed $50M",
                  "Election Day (5 Nov 2024)", "End of 2024"],
    "date": pd.to_datetime(["2023-12-31", "2024-03-19", first_day_over(25e6), first_day_over(50e6), "2024-11-05", "2024-12-31"])})
milestones = milestones.merge(lead[["date", "brown_cumulative", "moreno_cumulative", "lead", "ratio"]], on="date", how="left")
save_table(milestones, "lead_milestones.csv")
ms = milestones.set_index("milestone")
milestones

saved lead_milestones.csv (6, 6)


,milestone,date,brown_cumulative,moreno_cumulative,lead,ratio
0,End of 2023,2023-12-31,"9,622,203.73","5,252,610.92","4,369,592.81",1.83
1,Primary (19 Mar 2024),2024-03-19,"19,110,262.35","8,872,364.47","10,237,897.88",2.15
2,Lead first passed $25M,2024-06-10,"35,938,426.00","10,584,193.51","25,354,232.49",3.40
3,Lead first passed $50M,2024-09-23,"69,007,322.32","17,662,662.68","51,344,659.64",3.91
4,Election Day (5 Nov 2024),2024-11-05,"95,334,530.85","25,073,877.76","70,260,653.09",3.80
5,End of 2024,2024-12-31,"97,460,584.55","25,820,920.56","71,639,663.99",3.77


In [15]:
# Chart 3: Brown's lead over time
lead["ratio_text"] = lead["ratio"].map(lambda r: "n/a" if pd.isna(r) else format(r, ".2f"))
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lead["date"], y=lead["lead"] / 1e6, name="Brown's lead (Brown minus Moreno)", mode="lines",
    fill="tozeroy", line={"color": BROWN, "width": 2}, fillcolor="rgba(31, 95, 168, 0.25)",
    customdata=lead[["brown_cumulative", "moreno_cumulative", "ratio_text"]].to_numpy(),
    hovertemplate=("Lead: $%{y:,.2f} M<br>Brown so far: $%{customdata[0]:,.0f}<br>"
                   "Moreno so far: $%{customdata[1]:,.0f}<br>Brown has spent %{customdata[2]} times as much so far<extra></extra>")))

for name, text, ax, ay in [("Lead first passed $25M", "Passed $25M<br>(10 Jun 2024)", -10, -60),
                           ("Lead first passed $50M", "Passed $50M<br>(23 Sep 2024)", -30, -60),
                           ("Election Day (5 Nov 2024)", "Election Day lead:<br>$70.3M", 30, -55)]:
    row = ms.loc[name]
    fig.add_annotation(x=row["date"].strftime("%Y-%m-%d"), y=row["lead"] / 1e6, text=text, showarrow=True,
                       arrowhead=2, arrowcolor="#555555", ax=ax, ay=ay, font={"size": 11, "color": "#333333"},
                       bgcolor="#ffffff", bordercolor="#cccccc")
fig.update_layout(hovermode="x unified")
style_chart(fig, "Brown's lead grew from $4M at the end of 2023 to $70M on Election Day",
            "Cumulative total spent, Brown minus Moreno, campaign committees only, $ millions",
            xtitle="Date", ytitle="Brown's lead ($ millions)", money=True)
fig.update_yaxes(range=[0, lead["lead"].max() / 1e6 * 1.3])
fig.update_xaxes(range=["2022-12-01", "2025-01-15"])
add_key_dates(fig)
fig.write_html(OUT + "03_lead_over_time.html", include_plotlyjs="cdn")
fig.show()

## Metric 4: Share of each campaign's total spent in each period
**Question:** In which periods did each campaign spend its money?
**Definition:** For each candidate, the dollars paid in a phase divided by that candidate's own total (campaign committees). The five slices of each donut add up to 100%. The table also holds the dollars and the weekly pace (dollars divided by the number of weeks in the phase).

In [16]:
# Share of each candidate's spending by period, and weekly spending
period = campaign.groupby(["who", "phase"]).agg(total_spent=("amount", "sum"), payment_count=("amount", "size")).reset_index()
period = period.merge(phases, on="phase")
period["share_pct"] = period["total_spent"] / period.groupby("who")["total_spent"].transform("sum") * 100
period["weekly_pace"] = period["total_spent"] / (period["days"] / 7)
period = period.sort_values(["who", "start"]).rename(columns={"who": "candidate"})
period = period[["candidate", "phase", "start", "end", "days", "total_spent", "share_pct", "weekly_pace", "payment_count"]]
save_table(period, "period_summary.csv")
period

saved period_summary.csv (10, 9)


,candidate,phase,start,end,days,total_spent,share_pct,weekly_pace,payment_count
0,Brown,2023,2023-01-01,2023-12-31,365,"9,622,203.73",9.87,"184,535.41",868
2,Brown,"Jan 1 - Mar 19, 2024 (before the primary)",2024-01-01,2024-03-19,79,"9,488,058.62",9.74,"840,714.05",237
3,Brown,"Mar 20 - Aug 31, 2024",2024-03-20,2024-08-31,165,"37,845,564.90",38.83,"1,605,569.42",738
4,Brown,"Sep 1 - Nov 5, 2024 (final stretch)",2024-09-01,2024-11-05,66,"38,378,703.60",39.38,"4,070,468.56",548
1,Brown,After Election Day,2024-11-06,2024-12-31,56,"2,126,053.70",2.18,"265,756.71",126
5,Moreno,2023,2023-01-01,2023-12-31,365,"5,252,610.92",20.34,"100,735.00",802
7,Moreno,"Jan 1 - Mar 19, 2024 (before the primary)",2024-01-01,2024-03-19,79,"3,619,753.55",14.02,"320,737.66",285
8,Moreno,"Mar 20 - Aug 31, 2024",2024-03-20,2024-08-31,165,"6,063,573.14",23.48,"257,242.50",664
9,Moreno,"Sep 1 - Nov 5, 2024 (final stretch)",2024-09-01,2024-11-05,66,"10,137,940.15",39.26,"1,075,236.08",322
6,Moreno,After Election Day,2024-11-06,2024-12-31,56,"747,042.80",2.89,"93,380.35",110


In [17]:
# Chart 4: share of spending by period
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "domain"}]], horizontal_spacing=0.1)
for col, who in [(1, "Brown"), (2, "Moreno")]:
    rows = period[period["candidate"] == who]
    fig.add_trace(go.Pie(
        labels=rows["phase"], values=rows["total_spent"], hole=0.55, sort=False, direction="clockwise",
        marker={"colors": PHASE_COLORS, "line": {"color": "#ffffff", "width": 2}},
        text=[format(v, ".1f") + "%" for v in rows["share_pct"]], textinfo="text",
        textposition=["inside" if v >= 5 else "outside" for v in rows["share_pct"]], showlegend=(col == 1),
        hovertemplate="%{label}<br>$%{value:,.0f}<br>%{percent:.1%} of " + who + "'s total spent<extra></extra>"),
        row=1, col=col)
    fig.add_annotation(x=0.225 if col == 1 else 0.775, y=0.5, xref="paper", yref="paper", showarrow=False,
                       xanchor="center", yanchor="middle", text="<b>" + who + "</b>", font={"size": 20, "color": COLORS[who]})
style_chart(fig, "Moreno spent a bigger share of his money early; both spent about 39% in the final stretch",
            "Share of each campaign's own total spent, by period, campaign committees only", bottom=70, right=300)
fig.write_html(OUT + "04_share_by_period.html", include_plotlyjs="cdn")
fig.show()

## Metric 5: Average weekly spending in each period
**Question:** How fast did each campaign spend in each period?
**Definition:** Weekly spending = total spent in the phase divided by the number of weeks in the phase (days / 7), campaign committees. This corrects for the phases having different lengths (365, 79, 165, 66 and 56 days). The last phase (after Election Day) is short and is shown for completeness.

In [18]:
# Chart 5: weekly spending by period
fig = go.Figure()
for who in ["Brown", "Moreno"]:
    rows = period[period["candidate"] == who]
    fig.add_trace(go.Bar(
        x=[PHASE_WRAP[p] for p in rows["phase"]], y=rows["weekly_pace"] / 1e6, name=who, marker_color=COLORS[who],
        text=["$" + format(v / 1e6, ".2f") + " M" for v in rows["weekly_pace"]], textposition="outside", cliponaxis=False,
        customdata=rows[["phase", "weekly_pace", "total_spent", "days"]].to_numpy(),
        hovertemplate=("<b>" + who + "</b><br>%{customdata[0]}<br>$%{customdata[1]:,.0f} per week on average"
                       "<br>($%{customdata[2]:,.0f} over %{customdata[3]} days)<extra></extra>")))
fig.update_layout(barmode="group")
fast = period[period["phase"] == phases.loc[3, "phase"]].set_index("candidate")["weekly_pace"] / 1e6
style_chart(fig, "In the final stretch Brown spent $" + format(fast["Brown"], ".1f") + " M a week, Moreno $" + format(fast["Moreno"], ".1f") + " M",
            "Average weekly spending by period, campaign committees only, $ millions per week",
            xtitle="Period", ytitle="Average weekly spending ($ millions per week)", money=True)
fig.update_yaxes(range=[0, period["weekly_pace"].max() / 1e6 * 1.2])
fig.write_html(OUT + "05_weekly_pace.html", include_plotlyjs="cdn")
fig.show()

## Metric 6: Weekly spending before vs after a key date
**Question:** Did the weekly pace of spending change around key dates in the campaign?
**Definition:** Average weekly spending (campaign committees) in the **4 weeks after** a date (the date plus the next 27 days) compared with the **4 weeks before** it (the 28 days before the date). *Change* = after divided by before, minus 1. For **Election Day** the "after" window is the final 4 weeks (9 Oct - 5 Nov) and the "before" window is the 4 weeks before those (11 Sep - 8 Oct).

**Careful:** this only shows *when* money was paid. It does not show that the event caused the change, and the summer windows overlap (the conventions and Labor Day are only 2-5 weeks apart).

In [19]:
# Weekly spending in the 4 weeks before and after each key date (Election Day: last 4 weeks)
def weekly_changes(daily_table):
    rows = []
    for _, ev in key_dates.iterrows():
        t = ev["date"]
        if ev["event"] == "Election Day":
            after_start, after_end = t - pd.Timedelta(days=27), t
        else:
            after_start, after_end = t, t + pd.Timedelta(days=27)
        before_start, before_end = after_start - pd.Timedelta(days=28), after_start - pd.Timedelta(days=1)
        for who in ["Brown", "Moreno"]:
            before = daily_table.loc[before_start:before_end, who].sum() / 4
            after = daily_table.loc[after_start:after_end, who].sum() / 4
            rows.append({"event": ev["event"], "date": t, "candidate": who,
                         "before_start": before_start, "before_end": before_end, "after_start": after_start, "after_end": after_end,
                         "weekly_before": before, "weekly_after": after, "change_pct": (after / before - 1) * 100})
    return pd.DataFrame(rows)

key_change = weekly_changes(daily)
save_table(key_change, "key_dates_change.csv")
key_change[["event", "candidate", "weekly_before", "weekly_after", "change_pct"]]

saved key_dates_change.csv (10, 10)


,event,candidate,weekly_before,weekly_after,change_pct
0,Ohio primary,Brown,"1,277,391.05","1,323,146.47",3.58
1,Ohio primary,Moreno,"463,303.01","170,443.04",-63.21
2,Republican convention,Brown,"1,126,362.12","2,458,959.97",118.31
3,Republican convention,Moreno,"370,641.84","301,013.47",-18.79
4,Democratic convention,Brown,"1,794,732.46","2,722,258.69",51.68
5,Democratic convention,Moreno,"305,560.70","841,056.19",175.25
6,Labor Day,Brown,"2,196,471.63","3,714,173.10",69.10
7,Labor Day,Moreno,"507,689.08","882,140.79",73.76
8,Election Day,Brown,"3,911,438.20","4,235,747.63",8.29
9,Election Day,Moreno,"773,319.92","1,330,749.70",72.08


In [20]:
# Chart 6: before and after each key date
def pieces(event):
    rows = key_change[key_change["event"] == event].set_index("candidate")
    b, m = rows.loc["Brown"], rows.loc["Moreno"]
    title = (event + ": weekly spending Brown " + format(b["change_pct"], "+.1f") + "%, Moreno " + format(m["change_pct"], "+.1f") + "%")
    subtitle = ("Average weekly spending, campaign committees only, $ millions. After = " + b["after_start"].strftime("%d %b") + " to " + b["after_end"].strftime("%d %b %Y")
                + ", before = " + b["before_start"].strftime("%d %b") + " to " + b["before_end"].strftime("%d %b %Y"))
    y = [[b["weekly_before"] / 1e6, b["weekly_after"] / 1e6], [m["weekly_before"] / 1e6, m["weekly_after"] / 1e6]]
    text = [["$" + format(v, ".2f") + " M" for v in pair] for pair in y]
    exact = [[b["weekly_before"], b["weekly_after"]], [m["weekly_before"], m["weekly_after"]]]
    return title, subtitle, y, text, exact

first = pieces("Ohio primary")
fig = go.Figure()
for i, who in enumerate(["Brown", "Moreno"]):
    fig.add_trace(go.Bar(
        x=["4 weeks before", "4 weeks after"], y=first[2][i], name=who, marker_color=COLORS[who],
        text=first[3][i], textposition="outside", cliponaxis=False, customdata=first[4][i],
        hovertemplate="<b>" + who + "</b>, %{x}<br>$%{customdata:,.0f} per week on average<extra></extra>"))

buttons = []
for event in key_dates["event"]:
    t, s, y, text, exact = pieces(event)
    buttons.append({"label": event, "method": "update",
                    "args": [{"y": y, "text": text, "customdata": exact},
                             {"title.text": t + "<br><sup>" + s + "</sup>"}]})
fig.update_layout(barmode="group", updatemenus=[{
    "type": "dropdown", "buttons": buttons, "direction": "down", "showactive": True,
    "x": 0.08, "xanchor": "left", "y": 1.02, "yanchor": "bottom"}])
style_chart(fig, first[0], first[1], xtitle="Time window", ytitle="Average weekly spending ($ millions per week)", money=True)
fig.add_annotation(text="Key date:", xref="paper", yref="paper", x=0.075, y=1.02, xanchor="right", yanchor="bottom",
                   showarrow=False, font={"size": 12, "color": "#555555"}, yshift=8)
fig.write_html(OUT + "06_key_dates.html", include_plotlyjs="cdn")
fig.show()

## Metric 7: What the money was spent on, by period
**Question:** How did the mix of spending change from period to period?
**Definition:** For each candidate and phase, the share of dollars in three parts: **Voter communication** (TV/radio/other media, digital advertising, direct mail & printing, texting & phone, canvassing & field), **Fundraising costs**, and **Staff, overhead and other** (everything else). Campaign committees only. The three parts add up to 100% in each bar.

Note: categories come from keyword rules on free text, and Brown and Moreno label media buys differently, so we only compare these broad groups, never TV vs digital.

In [21]:
# Share of each spending type in each period
mix_wide = campaign.pivot_table(index=["who", "phase"], columns="spending_type", values="amount", aggfunc="sum", fill_value=0)
mix_wide["bar_total"] = mix_wide.sum(axis=1)
mix = mix_wide.reset_index().melt(id_vars=["who", "phase", "bar_total"], var_name="spending_type", value_name="amount")
mix["share_pct"] = mix["amount"] / mix["bar_total"] * 100
mix = mix.merge(phases[["phase", "start"]], on="phase").sort_values(["who", "start", "spending_type"]).rename(columns={"who": "candidate"})
mix = mix[["candidate", "phase", "spending_type", "amount", "share_pct"]]
save_table(mix, "mix_by_period.csv")
mix.pivot_table(index=["candidate", "phase"], columns="spending_type", values="share_pct").loc[["Brown", "Moreno"]].round(1)

saved mix_by_period.csv (30, 5)


spending_type                                        Fundraising costs  \
candidate phase                                                          
Brown     2023                                                   38.60   
          After Election Day                                      0.80   
          Jan 1 - Mar 19, 2024 (before the primary)               5.10   
          Mar 20 - Aug 31, 2024                                   0.20   
          Sep 1 - Nov 5, 2024 (final stretch)                     0.10   
Moreno    2023                                                   10.00   
          After Election Day                                      5.00   
          Jan 1 - Mar 19, 2024 (before the primary)               3.00   
          Mar 20 - Aug 31, 2024                                   5.80   
          Sep 1 - Nov 5, 2024 (final stretch)                     1.10   

spending_type                                        Staff, overhead and other  \
candidate phase                                                                  
Brown     2023                                                           31.40   
          After Election Day                                             57.70   
          Jan 1 - Mar 19, 2024 (before the primary)                      13.80   
          Mar 20 - Aug 31, 2024                                          10.70   
          Sep 1 - Nov 5, 2024 (final stretch)                             8.50   
Moreno    2023                                                           18.80   
          After Election Day                                             66.90   
          Jan 1 - Mar 19, 2024 (before the primary)                      10.20   
          Mar 20 - Aug 31, 2024                                          22.60   
          Sep 1 - Nov 5, 2024 (final stretch)                             8.80   

spending_type                                        Voter communication  
candidate phase                                                           
Brown     2023                                                     30.00  
          After Election Day                                       41.50  
          Jan 1 - Mar 19, 2024 (before the primary)                81.10  
          Mar 20 - Aug 31, 2024                                    89.10  
          Sep 1 - Nov 5, 2024 (final stretch)                      91.30  
Moreno    2023                                                     71.30  
          After Election Day                                       28.10  
          Jan 1 - Mar 19, 2024 (before the primary)                86.70  
          Mar 20 - Aug 31, 2024                                    71.70  
          Sep 1 - Nov 5, 2024 (final stretch)                      90.10

In [22]:
# Chart 7: spending mix by period
PART_COLORS = {"Voter communication": "#009E73", "Fundraising costs": "#E69F00", "Staff, overhead and other": "#999999"}
fig = make_subplots(rows=1, cols=2, subplot_titles=("Brown", "Moreno"), shared_yaxes=True, horizontal_spacing=0.06)
for col, who in [(1, "Brown"), (2, "Moreno")]:
    for part, color in PART_COLORS.items():
        rows = mix[(mix["candidate"] == who) & (mix["spending_type"] == part)]
        fig.add_trace(go.Bar(
            x=[PHASE_SHORT[p] for p in rows["phase"]], y=rows["share_pct"], name=part, marker_color=color,
            legendgroup=part, showlegend=(col == 1),
            text=[format(v, ".1f") + "%" if v >= 4 else "" for v in rows["share_pct"]], textposition="inside",
            customdata=rows[["phase", "amount"]].to_numpy(),
            hovertemplate=("<b>" + who + "</b>: " + part + "<br>%{customdata[0]}<br>%{y:.1f}% of the period's spending"
                           "<br>$%{customdata[1]:,.0f}<extra></extra>")), row=1, col=col)
fig.layout.annotations[0].update(font={"size": 16, "color": BROWN})
fig.layout.annotations[1].update(font={"size": 16, "color": MORENO})
fig.update_layout(barmode="stack", uniformtext={"minsize": 11, "mode": "hide"})
style_chart(fig, "In the 2024 campaign season, 72% to 91% of spending went to voter communication",
            "Share of each period's spending by type, campaign committees only, % of dollars",
            xtitle="Period", right=220)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.update_yaxes(title_text="Share of spending (%)", row=1, col=1)
fig.write_html(OUT + "07_spending_mix_by_period.html", include_plotlyjs="cdn")
fig.show()

## Metric 8: Share of spending by committee (all committees)
**Question:** How much of each side's spending came from the main campaign committee, and how much from the affiliated (joint fundraising / victory) committees?
**Definition:** Total spent by each of the nine committees as a % of the candidate's total across **all committees** (campaign + affiliated). Committees under 1% are merged into "Other committees". This is the only metric that shows affiliated committees on their own.

In [23]:
# Spending by committee as a share of the candidate's total
committee_share = df.groupby(["who", "committee", "group"])["amount"].sum().reset_index(name="total_spent")
committee_share["share_pct"] = committee_share["total_spent"] / committee_share.groupby("who")["total_spent"].transform("sum") * 100
committee_share["chart_slice"] = committee_share["committee"].where(committee_share["share_pct"] >= 1, "Other committees")
committee_share = committee_share.sort_values(["who", "total_spent"], ascending=[True, False]).rename(columns={"who": "candidate"})
save_table(committee_share, "committee_share.csv")
committee_share

saved committee_share.csv (9, 6)


,candidate,committee,group,total_spent,share_pct,chart_slice
1,Brown,Friends of Sherrod Brown,campaign,"97,460,584.55",97.01,Friends of Sherrod Brown
2,Brown,Ohio Grassroots Victory Fund,affiliated,"2,921,394.46",2.91,Ohio Grassroots Victory Fund
3,Brown,Senate Victory for Sherrod Brown 2024,affiliated,"60,599.39",0.06,Other committees
4,Brown,Victory Now for Sherrod Brown,affiliated,"22,732.82",0.02,Other committees
0,Brown,Canary Fund,affiliated,"1,943.91",0.00,Other committees
6,Moreno,Bernie Moreno for Senate,campaign,"25,805,693.28",85.67,Bernie Moreno for Senate
8,Moreno,Team Moreno,affiliated,"4,302,910.40",14.28,Team Moreno
7,Moreno,Moreno for OH Senate Republican Nominee Fund 2024,campaign,"13,190.77",0.04,Other committees
5,Moreno,Bernie Moreno for Ohio,campaign,"2,036.51",0.01,Other committees


In [24]:
# Chart 8: share of spending by committee
LIGHT = {"Brown": "#8fb3de", "Moreno": "#e69f98"}
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "domain"}]], horizontal_spacing=0.1)
for col, who in [(1, "Brown"), (2, "Moreno")]:
    rows = committee_share[committee_share["candidate"] == who]
    slices = rows.groupby("chart_slice").agg(total_spent=("total_spent", "sum"), group=("group", "first")).reset_index()
    slices["is_other"] = slices["chart_slice"] == "Other committees"
    slices = slices.sort_values(["is_other", "total_spent"], ascending=[True, False])
    colors = [("#9e9e9e" if o else (COLORS[who] if g == "campaign" else LIGHT[who])) for o, g in zip(slices["is_other"], slices["group"])]
    slice_pct = slices["total_spent"] / slices["total_spent"].sum() * 100
    fig.add_trace(go.Pie(
        labels=slices["chart_slice"], values=slices["total_spent"], hole=0.55, sort=False,
        marker={"colors": colors, "line": {"color": "#ffffff", "width": 2}},
        text=[format(v, ".1f") + "%" if v >= 1 else "" for v in slice_pct], textinfo="text",
        textposition=["inside" if v >= 8 else "outside" for v in slice_pct],
        hovertemplate="%{label}<br>$%{value:,.0f}<br>%{percent:.1%} of " + who + "'s all-committees total<extra></extra>"),
        row=1, col=col)
    fig.add_annotation(x=0.225 if col == 1 else 0.775, y=0.5, xref="paper", yref="paper", showarrow=False,
                       xanchor="center", yanchor="middle", text="<b>" + who + "</b><br>all committees", font={"size": 16, "color": COLORS[who]})
style_chart(fig, "The main campaign committee made up 97% of Brown's and 86% of Moreno's spending",
            "All committees (campaign + affiliated): share of each candidate's total spent; committees under 1% merged into 'Other committees'",
            bottom=70, right=250)
fig.write_html(OUT + "08_committee_share.html", include_plotlyjs="cdn")
fig.show()

## Metric 9: Big payments over time
**Question:** How many very large payments did each campaign make, and when?
**Definition:** One dot per payment of **more than $100,000** (campaign committees, `include == True`). A payment of exactly $100,000.00 is not counted (4 payments are exactly $100,000: 3 Brown, 1 Moreno).

In [25]:
# Payments over $100,000, the largest payment and payments over $1 million
big = campaign[campaign["amount"] > 100000].sort_values("date")
big = big.rename(columns={"who": "candidate_short"})[["candidate_short", "date", "payee", "category", "amount", "committee", "purpose", "phase"]]
big = big.rename(columns={"candidate_short": "candidate"})
save_table(big, "big_payments.csv")

largest = big.loc[big.groupby("candidate")["amount"].idxmax()]
over_1m = big[big["amount"] > 1_000_000].groupby("candidate")["amount"].agg(["size", "sum"])
over_1m["share_of_candidate_total_pct"] = over_1m["sum"] / camp["total_spent"] * 100
counts = big.groupby("candidate").size()
lg = largest.set_index("candidate")
print(largest[["candidate", "date", "payee", "amount"]].to_string(index=False))
over_1m

saved big_payments.csv (178, 8)
candidate       date                     payee       amount
    Brown 2024-09-16          GRASSROOTS MEDIA 2,652,813.00
   Moreno 2023-11-08 STRATEGIC MEDIA PLACEMENT 2,128,576.63


,size,sum,share_of_candidate_total_pct
candidate,,,
Brown,30,"43,654,747.86",44.79
Moreno,6,"8,034,197.63",31.12


In [26]:
# Chart 9: payments over $100,000
fig = go.Figure()
for who in ["Brown", "Moreno"]:
    rows = big[big["candidate"] == who]
    fig.add_trace(go.Scatter(
        x=rows["date"], y=rows["amount"] / 1e6, name=who, mode="markers",
        marker={"color": COLORS[who], "size": 9, "opacity": 0.75, "line": {"color": "#ffffff", "width": 0.5}},
        customdata=rows[["payee", "category", "amount"]].to_numpy(),
        hovertemplate=("<b>" + who + "</b>: %{customdata[0]}<br>%{customdata[1]}<br>"
                       "$%{customdata[2]:,.2f}<br>%{x|%d %b %Y}<extra></extra>")))

for who, ax, ay in [("Brown", -95, -35), ("Moreno", -125, -10)]:
    r = lg.loc[who]
    fig.add_annotation(x=r["date"].strftime("%Y-%m-%d"), y=r["amount"] / 1e6, showarrow=True, arrowhead=2, arrowcolor="#555555",
                       ax=ax, ay=ay, font={"size": 11, "color": COLORS[who]}, bgcolor="#ffffff", bordercolor="#cccccc",
                       text="Largest " + who + " payment<br>$" + format(r["amount"] / 1e6, ".2f") + " M to " + r["payee"].title() + "<br>" + r["date"].strftime("%d %b %Y"))
style_chart(fig, "Brown made " + str(counts["Brown"]) + " payments over $100,000; Moreno made " + str(counts["Moreno"]),
            "Each dot is one payment over $100,000, campaign committees only, $ millions",
            xtitle="Payment date", ytitle="Payment amount ($ millions)", money=True)
fig.update_yaxes(range=[0, big["amount"].max() / 1e6 * 1.3])
fig.update_xaxes(range=["2022-12-01", "2025-01-15"])
fig.update_layout(hovermode="closest")
add_key_dates(fig)
fig.write_html(OUT + "09_big_payments.html", include_plotlyjs="cdn")
fig.show()

## Dashboard-ready detail table
One row per candidate, committee, week (starting Monday), category and phase. A week that crosses a month or phase boundary is split into separate rows by date, so the sums stay exact. It includes both committee groups so the dashboard can filter by group.

In [27]:
# Weekly table used by the dashboard
detail = df.copy()
detail["week_start"] = detail["date"] - pd.to_timedelta(detail["date"].dt.dayofweek, unit="D")
detail = detail.groupby(["who", "candidate_id", "committee_id", "committee", "group", "week_start",
                         "month", "quarter", "phase", "spending_type", "category"]).agg(
    amount=("amount", "sum"), payment_count=("amount", "size")).reset_index().rename(columns={"who": "candidate", "group": "committee_group"})
save_table(detail, "spending_detail.csv")

detail.head()

saved spending_detail.csv (2179, 13)


,candidate,candidate_id,committee_id,committee,committee_group,week_start,month,quarter,phase,spending_type,category,amount,payment_count
0,Brown,S6OH00163,C00264697,Friends of Sherrod Brown,campaign,2022-12-26,2023-01,2023Q1,2023,"Staff, overhead and other",Processing & bank fees,55.84,1
1,Brown,S6OH00163,C00264697,Friends of Sherrod Brown,campaign,2023-01-02,2023-01,2023Q1,2023,"Staff, overhead and other",Payroll & benefits,60.60,1
2,Brown,S6OH00163,C00264697,Friends of Sherrod Brown,campaign,2023-01-02,2023-01,2023Q1,2023,"Staff, overhead and other",Processing & bank fees,453.32,8
3,Brown,S6OH00163,C00264697,Friends of Sherrod Brown,campaign,2023-01-02,2023-01,2023Q1,2023,"Staff, overhead and other",Reimbursements & card payments,"4,008.48",1
4,Brown,S6OH00163,C00264697,Friends of Sherrod Brown,campaign,2023-01-09,2023-01,2023Q1,2023,Fundraising costs,Fundraising costs,"5,000.00",1


## Sensitivity check: without the `possible_repeat` rows
`possible_repeat` marks look-alike payments (same committee, date, payee and amount). They are *flags*, not proven duplicates, so the main analysis keeps them. Here we remove them to see whether the story changes.

In [28]:
# Key-date changes without possible repeat rows
rep = campaign[campaign["possible_repeat"] == True]
print(len(rep), "possible_repeat rows in the campaign group, $" + format(rep["amount"].sum(), ",.0f"))

clean = campaign[campaign["possible_repeat"] == False]
daily_clean = clean.pivot_table(index="date", columns="who", values="amount", aggfunc="sum").reindex(pd.date_range("2023-01-01", "2024-12-31")).fillna(0)
key_change_clean = weekly_changes(daily_clean)

compare = key_change[["event", "candidate", "change_pct"]].merge(
    key_change_clean[["event", "candidate", "change_pct"]], on=["event", "candidate"], suffixes=("_with_repeats", "_without_repeats"))
compare["difference_points"] = compare["change_pct_without_repeats"] - compare["change_pct_with_repeats"]
print("Largest change in any key-date result (percentage points):", round(compare["difference_points"].abs().max(), 2))
compare

10 possible_repeat rows in the campaign group, $42,820
Largest change in any key-date result (percentage points): 0.11


,event,candidate,change_pct_with_repeats,change_pct_without_repeats,difference_points
0,Ohio primary,Brown,3.58,3.58,0.00
1,Ohio primary,Moreno,-63.21,-63.21,0.00
2,Republican convention,Brown,118.31,118.31,0.00
3,Republican convention,Moreno,-18.79,-18.78,0.00
4,Democratic convention,Brown,51.68,51.68,0.00
5,Democratic convention,Moreno,175.25,175.25,0.00
6,Labor Day,Brown,69.10,69.10,0.00
7,Labor Day,Moreno,73.76,73.76,0.00
8,Election Day,Brown,8.29,8.19,-0.11
9,Election Day,Moreno,72.08,72.08,0.00


In [29]:
# Totals and Election Day lead without possible repeat rows
total_clean = clean.groupby("who")["amount"].sum()
lead_clean = (daily_clean["Brown"].loc[:"2024-11-05"].sum() - daily_clean["Moreno"].loc[:"2024-11-05"].sum()) / 1e6
print("Totals without repeats ($M): Brown", round(total_clean["Brown"] / 1e6, 2), "| Moreno", round(total_clean["Moreno"] / 1e6, 2))
print("Election Day lead without repeats ($M):", round(lead_clean, 2), "(with repeats: 70.26)")

Totals without repeats ($M): Brown 97.42 | Moreno 25.82
Election Day lead without repeats ($M): 70.22 (with repeats: 70.26)


**What this shows:** Only 10 payments ($42,820) are flagged as possible repeats, and removing them changes the results very little (for example Brown's Election Day change goes from +8.3% to +8.2%). The story does not depend on them.

## Validation
Every check compares a result with a reference number, or with the saved table. All checks must pass.

In [30]:
# Checks on totals and committees
checks = []
def check(name, expected, actual, digits=None, tolerance=0.005):
    if isinstance(expected, str):
        ok = (expected == actual)
    else:
        if digits is not None:
            actual = round(actual, digits)
            tolerance = 1e-9
        ok = abs(expected - actual) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "result": "PASS" if ok else "FAIL"})
    print("PASS" if ok else "FAIL", "-", name, "| expected:", expected, "| actual:", actual)

check("Included rows", 5912, len(df))
check("All rows in the file", 10718, len(spend))

tot = df.groupby(["who", "group"])["amount"].sum()
check("Brown, campaign group total", 97460584.55, tot["Brown", "campaign"], digits=2)
check("Moreno, campaign group total", 25820920.56, tot["Moreno", "campaign"], digits=2)
check("Brown, affiliated total", 3006670.58, tot["Brown", "affiliated"], digits=2)
check("Moreno, affiliated total", 4302910.40, tot["Moreno", "affiliated"], digits=2)
check("Brown, all committees", 100467255.13, tot["Brown"].sum(), digits=2)
check("Moreno, all committees", 30123830.96, tot["Moreno"].sum(), digits=2)
check("Brown spent this many times as much (campaign)", 3.8, tot["Brown", "campaign"] / tot["Moreno", "campaign"], digits=1)

by_committee = df.groupby("committee")["amount"].sum()
expected_by_committee = {
    "Friends of Sherrod Brown": 97460584.55, "Ohio Grassroots Victory Fund": 2921394.46,
    "Senate Victory for Sherrod Brown 2024": 60599.39, "Victory Now for Sherrod Brown": 22732.82,
    "Canary Fund": 1943.91, "Bernie Moreno for Senate": 25805693.28, "Team Moreno": 4302910.40,
    "Moreno for OH Senate Republican Nominee Fund 2024": 13190.77, "Bernie Moreno for Ohio": 2036.51}
for name, value in expected_by_committee.items():
    check("Committee total: " + name, value, by_committee[name], digits=2)

check("Committees in dim_committees.csv", 9, len(committees))
check("Every spending committee is in dim_committees.csv", "True", str(set(df["committee_id"]) <= set(committees["committee_id"])))

check("Moreno for Senate vs official FEC total (exact match)", 25805693.28, by_committee["Bernie Moreno for Senate"], digits=2)
check("Friends of Sherrod Brown is below the official FEC total by", 6699.03, 97467283.58 - by_committee["Friends of Sherrod Brown"], digits=2)

PASS - Included rows | expected: 5912 | actual: 5912
PASS - All rows in the file | expected: 10718 | actual: 10718
PASS - Brown, campaign group total | expected: 97460584.55 | actual: 97460584.55
PASS - Moreno, campaign group total | expected: 25820920.56 | actual: 25820920.56
PASS - Brown, affiliated total | expected: 3006670.58 | actual: 3006670.58
PASS - Moreno, affiliated total | expected: 4302910.4 | actual: 4302910.4
PASS - Brown, all committees | expected: 100467255.13 | actual: 100467255.13
PASS - Moreno, all committees | expected: 30123830.96 | actual: 30123830.96
PASS - Brown spent this many times as much (campaign) | expected: 3.8 | actual: 3.8
PASS - Committee total: Friends of Sherrod Brown | expected: 97460584.55 | actual: 97460584.55
PASS - Committee total: Ohio Grassroots Victory Fund | expected: 2921394.46 | actual: 2921394.46
PASS - Committee total: Senate Victory for Sherrod Brown 2024 | expected: 60599.39 | actual: 60599.39
PASS - Committee total: Victory Now for Sh

In [31]:
# Checks on each metric
check("Metric 1: Brown campaign total", 97460584.55, camp.loc["Brown", "total_spent"], digits=2)
check("Metric 1: Moreno campaign total", 25820920.56, camp.loc["Moreno", "total_spent"], digits=2)
check("Metric 1: ratio rounded to 1 decimal", 3.8, ratio_total, digits=1)

ref_brown_q = [1.2, 2.1, 3.2, 3.1, 10.6, 17.8, 36.3, 23.2]
ref_moreno_q = [0.0, 0.7, 0.7, 3.8, 3.9, 1.7, 7.7, 7.3]
for q, b, m in zip(all_quarters, ref_brown_q, ref_moreno_q):
    check("Metric 2: " + q + " Brown ($M)", b, quarterly.loc[q, "Brown"] / 1e6, digits=1)
    check("Metric 2: " + q + " Moreno ($M)", m, quarterly.loc[q, "Moreno"] / 1e6, digits=1)
check("Metric 2: Brown's biggest month", "2024-10", monthly["Brown"].idxmax())
check("Metric 2: Moreno's biggest month", "2024-10", monthly["Moreno"].idxmax())
check("Metric 2: Brown October 2024 ($M)", 19.16, monthly.loc["2024-10", "Brown"] / 1e6, digits=2)
check("Metric 2: Moreno October 2024 ($M)", 6.30, monthly.loc["2024-10", "Moreno"] / 1e6, digits=2)
months_2024 = monthly.loc["2024-01":"2024-12"]
check("Metric 2: Brown out-spent Moreno in every month of 2024", "True", str((months_2024["Brown"] > months_2024["Moreno"]).all()))

for name, lead_ref, ratio_ref in [("End of 2023", 4.37, 1.83), ("Primary (19 Mar 2024)", 10.24, 2.15),
                                  ("Election Day (5 Nov 2024)", 70.26, 3.80), ("End of 2024", 71.64, 3.77)]:
    check("Metric 3: lead at " + name + " ($M)", lead_ref, ms.loc[name, "lead"] / 1e6, digits=2)
    check("Metric 3: ratio at " + name, ratio_ref, ms.loc[name, "ratio"], digits=2)
check("Metric 3: lead first passed $25M on", "2024-06-10", ms.loc["Lead first passed $25M", "date"].strftime("%Y-%m-%d"))
check("Metric 3: lead first passed $50M on", "2024-09-23", ms.loc["Lead first passed $50M", "date"].strftime("%Y-%m-%d"))

ref_share = {"Brown": [9.9, 9.7, 38.8, 39.4, 2.2], "Moreno": [20.3, 14.0, 23.5, 39.3, 2.9]}
ref_pace = {"Brown": [0.18, 0.84, 1.61, 4.07], "Moreno": [0.10, 0.32, 0.26, 1.08]}
for who in ["Brown", "Moreno"]:
    rows = period[period["candidate"] == who].reset_index(drop=True)
    for i, letter in enumerate("ABCDE"):
        check("Metric 4: " + who + " share of own total, phase " + letter + " (%)", ref_share[who][i], rows.loc[i, "share_pct"], digits=1)
    for i, letter in enumerate("ABCD"):
        check("Metric 5: " + who + " weekly spending, phase " + letter + " ($M)", ref_pace[who][i], rows.loc[i, "weekly_pace"] / 1e6, digits=2)
check("Phase lengths in days (A-D)", "365,79,165,66", ",".join(str(d) for d in phases["days"][:4]))

ref_change = {"Ohio primary": (3.6, -63.2), "Republican convention": (118.3, -18.8), "Democratic convention": (51.7, 175.3),
              "Labor Day": (69.1, 73.8), "Election Day": (8.3, 72.1)}
kc = key_change.set_index(["event", "candidate"])
for event, (b, m) in ref_change.items():
    check("Metric 6: " + event + ", Brown change (%)", b, kc.loc[(event, "Brown"), "change_pct"], digits=1)
    check("Metric 6: " + event + ", Moreno change (%)", m, kc.loc[(event, "Moreno"), "change_pct"], digits=1)

ref_mix = {"Brown": [(30.0, 38.6), (81.1, 5.1), (89.1, 0.2), (91.3, 0.1), (41.5, 0.8)],
           "Moreno": [(71.3, 10.0), (86.7, 3.0), (71.7, 5.8), (90.1, 1.1), (28.1, 5.0)]}
mix_idx = mix.set_index(["candidate", "phase", "spending_type"])["share_pct"]
for who in ["Brown", "Moreno"]:
    for p, (voter, fund) in zip(phases["phase"], ref_mix[who]):
        check("Metric 7: " + who + ", " + p + ", voter communication (%)", voter, mix_idx[who, p, "Voter communication"], digits=1)
        check("Metric 7: " + who + ", " + p + ", fundraising costs (%)", fund, mix_idx[who, p, "Fundraising costs"], digits=1)

sums = committee_share.groupby("candidate")["share_pct"].sum()
check("Metric 8: Brown shares add up to 100%", 100.0, sums["Brown"], digits=6)
check("Metric 8: Moreno shares add up to 100%", 100.0, sums["Moreno"], digits=6)
check("Metric 8: slices left for Brown", 3, committee_share[committee_share["candidate"] == "Brown"]["chart_slice"].nunique())
check("Metric 8: slices left for Moreno", 3, committee_share[committee_share["candidate"] == "Moreno"]["chart_slice"].nunique())

check("Metric 9: Brown payments over $100,000", 143, counts["Brown"])
check("Metric 9: Moreno payments over $100,000", 35, counts["Moreno"])
check("Metric 9: Brown largest payment", 2652813.00, lg.loc["Brown", "amount"], digits=2)
check("Metric 9: Brown largest payment date", "2024-09-16", lg.loc["Brown", "date"].strftime("%Y-%m-%d"))
check("Metric 9: Brown largest payment payee", "GRASSROOTS MEDIA", lg.loc["Brown", "payee"].upper())
check("Metric 9: Moreno largest payment", 2128576.63, lg.loc["Moreno", "amount"], digits=2)
check("Metric 9: Moreno largest payment date", "2023-11-08", lg.loc["Moreno", "date"].strftime("%Y-%m-%d"))
check("Metric 9: Moreno largest payment payee", "STRATEGIC MEDIA PLACEMENT", lg.loc["Moreno", "payee"].upper())
check("Metric 9: Brown payments over $1M", 30, over_1m.loc["Brown", "size"])
check("Metric 9: Moreno payments over $1M", 6, over_1m.loc["Moreno", "size"])
check("Metric 9: Brown share of own spending in payments over $1M (%)", 44.8, over_1m.loc["Brown", "share_of_candidate_total_pct"], digits=1)
check("Metric 9: Moreno share of own spending in payments over $1M (%)", 31.1, over_1m.loc["Moreno", "share_of_candidate_total_pct"], digits=1)

PASS - Metric 1: Brown campaign total | expected: 97460584.55 | actual: 97460584.55
PASS - Metric 1: Moreno campaign total | expected: 25820920.56 | actual: 25820920.56
PASS - Metric 1: ratio rounded to 1 decimal | expected: 3.8 | actual: 3.8
PASS - Metric 2: 2023Q1 Brown ($M) | expected: 1.2 | actual: 1.2
PASS - Metric 2: 2023Q1 Moreno ($M) | expected: 0.0 | actual: 0.0
PASS - Metric 2: 2023Q2 Brown ($M) | expected: 2.1 | actual: 2.1
PASS - Metric 2: 2023Q2 Moreno ($M) | expected: 0.7 | actual: 0.7
PASS - Metric 2: 2023Q3 Brown ($M) | expected: 3.2 | actual: 3.2
PASS - Metric 2: 2023Q3 Moreno ($M) | expected: 0.7 | actual: 0.7
PASS - Metric 2: 2023Q4 Brown ($M) | expected: 3.1 | actual: 3.1
PASS - Metric 2: 2023Q4 Moreno ($M) | expected: 3.8 | actual: 3.8
PASS - Metric 2: 2024Q1 Brown ($M) | expected: 10.6 | actual: 10.6
PASS - Metric 2: 2024Q1 Moreno ($M) | expected: 3.9 | actual: 3.9
PASS - Metric 2: 2024Q2 Brown ($M) | expected: 17.8 | actual: 17.8
PASS - Metric 2: 2024Q2 Moreno ($

In [32]:
# Checks on the dashboard table and the sensitivity result
check("Detail table: total of all rows", 130591086.09, detail["amount"].sum(), digits=2)
check("Detail table: number of payments", 5912, detail["payment_count"].sum())
d_camp = detail[detail["committee_group"] == "campaign"].groupby("candidate")["amount"].sum()
check("Detail table: Brown campaign total", 97460584.55, d_camp["Brown"], digits=2)
check("Detail table: Moreno campaign total", 25820920.56, d_camp["Moreno"], digits=2)

check("Sensitivity: possible_repeat rows in the campaign group", 10, len(rep))
check("Sensitivity: possible_repeat dollars", 42820.0, rep["amount"].sum(), digits=0)
check("Sensitivity: Brown Election Day change without repeats (%)", 8.2,
      key_change_clean.set_index(["event", "candidate"]).loc[("Election Day", "Brown"), "change_pct"], digits=1)

PASS - Detail table: total of all rows | expected: 130591086.09 | actual: 130591086.09
PASS - Detail table: number of payments | expected: 5912 | actual: 5912
PASS - Detail table: Brown campaign total | expected: 97460584.55 | actual: 97460584.55
PASS - Detail table: Moreno campaign total | expected: 25820920.56 | actual: 25820920.56
PASS - Sensitivity: possible_repeat rows in the campaign group | expected: 10 | actual: 10
PASS - Sensitivity: possible_repeat dollars | expected: 42820.0 | actual: 42820.0
PASS - Sensitivity: Brown Election Day change without repeats (%) | expected: 8.2 | actual: 8.2


In [33]:
# Save all checks and stop if any failed
validation = pd.DataFrame(checks)
validation["result"] = validation["result"].str.lower()
validation = validation.rename(columns={"result": "pass_fail"})
save_table(validation, "validation_checks.csv")
print(validation["pass_fail"].value_counts().to_dict())
assert (validation["pass_fail"] == "pass").all(), "Some checks FAILED: see the list above"
print("All checks passed.")

saved validation_checks.csv (128, 4)
{'pass': 128}
All checks passed.
